# Git individual desde Colab

Ejecuta una celda cada vez y comprueba el resultado. Usa tu propio repositorio público `mi-practica-git`, creado con README. Para guardar solo este notebook, usa File → Save a copy in GitHub. Para practicar con los archivos del repositorio, sigue las celdas. No incluyas tokens en el código ni en las salidas.

In [ ]:
!git --version

## 1. Clonar
Sustituye `TU_USUARIO` antes de ejecutar. Ejecuta clone una sola vez por sesión. Si ya existe la carpeta, no la borres: entra en ella y revisa su estado.

In [ ]:
!git clone https://github.com/TU_USUARIO/mi-practica-git.git
%cd /content/mi-practica-git
!git status
!git branch --show-current
!git remote -v

## 2. Configurar la autoría
Sustituye los datos por tu nombre y un correo verificado o el noreply exacto de GitHub. Esta configuración no inicia sesión.

In [ ]:
!git config user.name "Tu nombre"
!git config user.email "tu-correo-asociado-a-github@example.com"
!git pull --ff-only origin main

## 3. Crear un archivo
La celda evita sobrescribir un archivo existente. Si repites la práctica, edita el archivo desde el panel Files o usa otro nombre y ajusta los comandos siguientes.

In [ ]:
from pathlib import Path
nota = Path("notas-colab.md")
with nota.open("x", encoding="utf-8") as archivo:
    archivo.write("# Mi práctica en Colab\n\nHe aprendido a crear un commit desde un notebook.\n")
print(nota.read_text(encoding="utf-8"))

## 4. Add y commit
Revisa el contenido preparado antes de confirmar.

In [ ]:
!git status
!git add notas-colab.md
!git diff --staged

In [ ]:
!git commit -m "Añade notas de la práctica en Colab"
!git log --oneline -3

## 5. Publicar con autenticación temporal
Crea en GitHub un fine-grained personal access token, limitado a este repositorio, con Contents: Read and write y vencimiento breve. Introduce el token solo en el campo oculto cuando se solicite. Usa una sesión personal de confianza y revócalo al terminar. El código desactiva el almacenamiento de credenciales para esta operación y elimina el programa temporal en un bloque finally. Un push fallido no borra tu commit: revisa el error antes de repetir. Nunca pongas el token en la URL.

In [ ]:
import getpass
import os
import subprocess
import tempfile
from pathlib import Path

usuario = input("Tu usuario de GitHub: ").strip()
token = getpass.getpass("Token temporal (oculto): ")
try:
    with tempfile.TemporaryDirectory() as carpeta_temporal:
        askpass = Path(carpeta_temporal) / "askpass.py"
        askpass.write_text(
            "#!/usr/bin/env python3\n"
            "import os, sys\n"
            "prompt = sys.argv[1].lower() if len(sys.argv) > 1 else ''\n"
            "print(os.environ['PRACTICA_GH_USER'] if 'username' in prompt else os.environ['PRACTICA_GH_TOKEN'])\n",
            encoding="utf-8",
        )
        askpass.chmod(0o700)
        entorno = os.environ.copy()
        entorno.update({
            "GIT_ASKPASS": str(askpass),
            "GIT_TERMINAL_PROMPT": "0",
            "PRACTICA_GH_USER": usuario,
            "PRACTICA_GH_TOKEN": token,
        })
        try:
            subprocess.run(
                ["git", "-c", "credential.helper=", "push", "origin", "main"],
                env=entorno,
                check=True,
            )
        finally:
            entorno.pop("PRACTICA_GH_TOKEN", None)
finally:
    token = None


## 6. Comprobar
Abre tu repositorio en GitHub y verifica el archivo y el commit. El estado limpio indica que no quedan cambios locales sin confirmar; comprueba también que push terminó correctamente.

In [ ]:
!git status
!git log --oneline -5

## 7. Recuperar versiones
Para practicar revert, crea primero un commit separado con un error deliberado y publícalo. Revisa que HEAD sea ese commit y ejecuta `!git revert --no-edit HEAD`. Después vuelve a ejecutar la celda de publicación. No reviertas un commit al azar. Consulta `docs/05-recuperar-versiones.md`.

Los archivos de `/content` son temporales. Guardar este notebook en Drive no guarda el clon. Publica y verifica tus commits antes de cerrar. Puedes borrar las salidas antes de guardar el notebook en GitHub.